> **What you already know:** Keras `model.compile()`, `model.fit()`, `model.evaluate()`, `save_weights`/`load_weights` (from `00-pytorch-primer`).
> **The gap this chapter closes:** You cannot yet write a manual training loop with `tf.GradientTape`, or build a network that processes a sequence of steps.
> **What you'll have by the end:** A trained `HousePriceModel` with GradientTape-verified gradients, SimpleRNN and LSTM models for sequence prediction.
> **What this chapter is not:** A production RNN implementation — that is `01-rnns/` Part 2.

<table align="center">
  <td align="center"><a target="_blank" href="http://introtodeeplearning.com">
        <img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
      Visit MIT Deep Learning</a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com">
        <img src="https://i.ibb.co/2P3SLwK/colab.png" style="padding-bottom:5px;" />Run in Google Colab</a></td>
</table>

# Copyright Information

In [ ]:
# Copyright 2026 MIT Introduction to Deep Learning. All Rights Reserved.
#
# Licensed under the MIT License. You may not use this file except in compliance
# with the License. Use and/or modification of this code outside of MIT Introduction
# to Deep Learning must reference:
#
# © MIT Introduction to Deep Learning
# http://introtodeeplearning.com
#

# TensorFlow/Keras from First Principles

## Building Deep Learning Intuition One Tensor at a Time

This notebook builds the **complete mental model for TensorFlow's tensor abstraction and
`tf.GradientTape` automatic differentiation system** — starting with a running example
that threads through every concept.

Every concept is demonstrated on the same problem:

> **Predicting house prices from size and age**
> 5 houses, 2 features (sq ft, age) → 1 price. Small enough to visualise, real enough to matter.

| Part | Concept | Key Idea |
| ---- | ------- | -------- |
| 1 | Tensors as Data Containers | Scalars → vectors → matrices → batches; faster + safer than lists |
| 2 | Computations on Tensors | TF traces operations; hand-tuning weights fails at scale |
| 3 | Neural Networks in Keras | `tf.keras.layers.Layer` wraps `add_weight`; `call()` defines the computation |
| 4 | Automatic Differentiation | `tf.GradientTape` computes ∂loss/∂every_weight in one call; gradient descent converges |
| 5 | From Toy to Production | Same GradientTape scales from 3 params to 25 million (ResNet50) |
| 6 | Sequence Modeling with RNN/LSTM | `layers.SimpleRNN` and `layers.LSTM` extend the same pattern to ordered sequences |

---

## 0. Setup

[TensorFlow](https://www.tensorflow.org/) / [Keras](https://keras.io/) is a deep learning
library known for its ease of use and production-readiness. This notebook uses TensorFlow
with the Keras high-level API.

---

## Prerequisite Bridge — From `00-pytorch-primer`

| Foundation (from `00-pytorch-primer/keras-to-pytorch-primer.ipynb`) | Role in this notebook |
| -------------------------------------------------------------------- | --------------------- |
| `keras.Sequential` + `model.compile()` + `model.fit()` | Used for comparison; this notebook goes lower: `tf.GradientTape` instead of `fit()` |
| `model.evaluate()` / `model.predict()` | The high-level API; we build what's underneath it here |
| Channels-last `(N, H, W, C)` shape convention | Carried through all Keras layer definitions |
| `save_weights` / `load_weights` round-trip | Same pattern works for models built here |

> **If you haven't completed `00-pytorch-primer/keras-to-pytorch-primer.ipynb`** the `keras.layers` and
> `model.compile()` patterns used here will feel unfamiliar. That notebook takes 30–40 minutes.

## Table of Contents

1. [Where This Notebook Sits in the Larger Topic Space](#where-this-notebook-sits)
2. [Setup](#0-setup)
3. [Part 1 — Tensors as Data Containers](#part-1--tensors-as-data-containers)
4. [Part 2 — Computations on Tensors](#part-2--computations-on-tensors)
5. [Part 3 — Neural Networks in Keras](#part-3--neural-networks-in-keras)
6. [Part 4 — Automatic Differentiation with GradientTape](#part-4--automatic-differentiation)
   - [Gradient Descent on a Parabola](#code-walkthrough-gradient-descent-on-a-parabola)
   - [Training Loop to Convergence](#training-loop-to-convergence)
7. [Part 5 — From Toy to Production](#part-5--from-toy-to-production)
8. [Part 6 — Sequence Modeling: SimpleRNN and LSTM](#part-6--sequence-modeling)
9. [What This Notebook Covered](#what-this-notebook-covered)
10. [Summary — What You Built](#summary--what-you-built)

### Where This Notebook Sits in the Larger Topic Space

| Topic area | Covered here | Deliberately out of scope |
| ---------- | ------------ | ------------------------- |
| Tensor mechanics | Creation, shape/`ndim`, indexing, NumPy interop | Reshaping, broadcasting rules, dtype casting |
| Autograd | `tf.GradientTape`, `tape.gradient()`, gradient inspection | Custom gradients, higher-order gradients |
| Model building | `add_weight`, `tf.keras.Sequential`, subclassing `Layer` | Full `tf.data` pipelines, custom callbacks |
| Training loop | Full loop to convergence with a real loss curve | Mini-batching via `tf.data`, regularization, LR scheduling |
| Production scale | Real parameter counts (ResNet50) | Fine-tuning, mixed precision, deployment/export |
| Sequence modeling | `layers.SimpleRNN`, `layers.LSTM`, sine-wave prediction | Attention, Transformer, BPTT depth analysis |

In [ ]:
#  Install dependencies (run once)
import subprocess, sys

required = [
    ("numpy",      "numpy"),
    ("matplotlib", "matplotlib"),
    ("tensorflow", "tensorflow"),
]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#  Deterministic seeds for reproducible results
tf.random.set_seed(42)
np.random.seed(42)

gpus = tf.config.list_physical_devices("GPU")
device_str = "GPU" if gpus else "CPU"
print("Seeds set — every run produces identical results.")
print(f"TensorFlow will use: {device_str}")
print("Note: unlike PyTorch, TF uses GPU automatically — no .to(device) calls needed.")

### Predict before you run

Before running the scatter plot below, commit to an answer: which feature will show
the stronger visual correlation with house price?

1. **Size** — bigger houses always cost more; age barely matters.
2. **Age** — newer houses cost more; size is secondary.
3. **Both roughly equally** — two similarly strong linear trends.

In [ ]:
#  Our Running Example — House Price Prediction
houses = tf.constant(
    [
        [1200.0, 10.0],  # size (sq ft), age (years)
        [1500.0,  5.0],
        [ 800.0, 15.0],
        [2000.0,  2.0],
        [1000.0, 12.0],
    ],
    dtype=tf.float32,
)
prices = tf.constant([250.0, 320.0, 180.0, 450.0, 210.0])

print("Our running example — 5 houses:")
print("  Size (sq ft)  Age (yrs)  Price ($k)")
for i, (h, p) in enumerate(zip(houses.numpy(), prices.numpy())):
    print(f"  [{i}]  {h[0]:6.0f}      {h[1]:4.0f}       {p:6.0f}")

In [ ]:
#  2-panel scatter plot
h_np = houses.numpy()
p_np = prices.numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sc0 = axes[0].scatter(h_np[:, 0], p_np, c=h_np[:, 1], cmap="viridis", s=100, edgecolor="black")
axes[0].set_xlabel("Size (sq ft)")
axes[0].set_ylabel("Price ($1000s)")
axes[0].set_title("Price vs Size (color = age)")
plt.colorbar(sc0, ax=axes[0], label="Age (years)")
sc1 = axes[1].scatter(h_np[:, 1], p_np, c=h_np[:, 0], cmap="plasma", s=100, edgecolor="black")
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Price ($1000s)")
axes[1].set_title("Price vs Age (color = size)")
plt.colorbar(sc1, ax=axes[1], label="Size (sq ft)")
plt.suptitle("Our Running Example: 5 Houses → 1 Price", fontweight="bold")
plt.tight_layout()
plt.show()
print("This dataset is our 'cat sat on the mat' — every concept demonstrated on these 5 houses.")

#### What just happened — and what's missing

We established the running example: 5 houses, 2 input features (size, age), 1 price label.
The scatter plots confirm size correlates positively with price while age correlates negatively.

What's missing: we have raw Python data but no structure that can batch, broadcast, or
GPU-accelerate it. For that we need **tensors** — the topic of Part 1.

---

## Part 1 — Tensors as Data Containers

TensorFlow is a machine learning library. At its core it provides an interface for creating
and manipulating **tensors** — multi-dimensional arrays. Like PyTorch tensors, TF tensors are:

- Fast (backed by optimized BLAS kernels, GPU-capable)
- Safe (shape-checked at every operation)
- Purpose-built (every higher-level Keras construct is built on `tf.Tensor`)

Key difference from PyTorch: `tf.constant` creates an **immutable** tensor (like a frozen
NumPy array), while `tf.Variable` creates a **mutable** one — used for learnable parameters.
PyTorch uses `torch.tensor` for data and `nn.Parameter` for learnable weights.

The `shape` and `ndim` attributes work identically to PyTorch.

#### **Predict first** — why do we need tensors?

Python has lists. NumPy has arrays. Predict which of these are true about `tf.Tensor` vs
nested lists:

1. **Speed**: Can tensors outrun nested lists on matrix multiply?
2. **Shape safety**: Will tensors catch a shape mismatch that lists silently swallow?
3. **GPU support**: Can a list of lists run on a GPU?

Make your prediction, then run the next three cells — one measured proof per reason.

In [ ]:
#  Reason 1: Speed — matrix multiply
import time

n = 1000
np_a = np.random.randn(n, n).astype(np.float32)
np_b = np.random.randn(n, n).astype(np.float32)
t0 = time.time()
_ = np_a @ np_b
np_time = time.time() - t0

tf_a = tf.constant(np_a)
tf_b = tf.constant(np_b)
_ = tf.linalg.matmul(tf_a, tf_b)  # warm up
t0 = time.time()
_ = tf.linalg.matmul(tf_a, tf_b)
tf_time = time.time() - t0

print(f"Matrix multiply (1000×1000):")
print(f"  NumPy:       {np_time*1000:.1f} ms")
print(f"  TensorFlow:  {tf_time*1000:.1f} ms")
print("  Lists would use interpreted loops — orders of magnitude slower.")

**Reason 2 — shape safety.** Nested Python lists don't check shapes at all. Does
`tf.Tensor` catch a shape mismatch immediately, at the point of the mistake?

In [ ]:
#  Reason 2: Shape safety
try:
    bad = tf.constant([[1.0, 2.0]]) + tf.constant([[1.0, 2.0, 3.0]])
except Exception as e:
    print(f"Shape mismatch caught: {type(e).__name__}")
    print("  tf.Tensor validates shapes at every operation and raises immediately.")
    print("  Lists would silently fail or produce a confusing IndexError elsewhere.")

**Reason 3 — GPU support.** Deep learning workloads only run at scale because they use
GPUs. Can a plain Python list of lists move to a GPU the way a tensor can?

In [ ]:
#  Reason 3: GPU support
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU available: {len(gpus) > 0}")
print("  → tf.Tensor operations run on GPU automatically when one is present.")
print("  → Lists cannot.")
print("\n→ Tensors are PURPOSE-BUILT for deep learning: fast, safe, GPU-ready.")

### Code Walkthrough: Three Reasons Tensors Beat Python Lists

**`tf.constant(np_a)` — zero-overhead bridge from NumPy**
Converts a NumPy array to a TF constant sharing similar memory layout. Subsequent math
dispatches to optimized BLAS kernels.

**`tf.constant([[1.0, 2.0]]) + tf.constant([[1.0, 2.0, 3.0]])` → exception**
TF validates shapes at every operation and raises immediately. The error is at the exact
point of the mistake — not 10 lines later.

**`tf.config.list_physical_devices('GPU')`**
Lists available GPU devices. TF automatically routes tensor operations to GPU when one is
present — no explicit `.to(device)` calls like PyTorch requires.

> **Key takeaway:** TF tensors are purpose-built for deep learning — fast, shape-safe, and
> GPU-portable. Everything in Keras is built on this foundation.

In [ ]:
#  Scalar tensors (0-D)
integer = tf.constant(1234)
decimal = tf.constant(3.14159265359)

print(f"`integer` is a {integer.ndim}-d Tensor: {integer.numpy()}")
print(f"`decimal` is a {decimal.ndim}-d Tensor: {decimal.numpy():.5f}")

Vectors and lists can be used to create 1-d tensors:

In [ ]:
#  1-D Tensors (vectors)
fibonacci = tf.constant([1, 1, 2, 3, 5, 8])
count_to_100 = tf.constant(list(range(100)))

print(f"`fibonacci` is a {fibonacci.ndim}-d Tensor with shape: {fibonacci.shape}")
print(f"`count_to_100` is a {count_to_100.ndim}-d Tensor with shape: {count_to_100.shape}")

Next, let's create 2-D (matrices) and higher-rank tensors. In Keras, images use 4-D tensors
with dimensions `(batch, height, width, channels)` — channels last (NHWC), opposite of
PyTorch's `(batch, channels, height, width)` (NCHW).

In [ ]:
#  2-D and higher-rank Tensors
matrix = tf.constant([[1, 2, 3], [4, 5, 6]])
assert matrix.ndim == 2

# Keras convention: (batch, height, width, channels) — channels LAST (NHWC)
# Compare with PyTorch: (batch, channels, height, width) — channels FIRST (NCHW)
images = tf.zeros([10, 256, 256, 3])  # 10 images, 256×256, RGB — Keras NHWC format

assert images.ndim == 4
assert images.shape == (10, 256, 256, 3)
print(f"images is a {images.ndim}-d Tensor with shape: {images.shape}")
print("Keras: (batch, H, W, C) — channels last. PyTorch: (batch, C, H, W) — channels first.")

As in PyTorch, `shape` gives the number of elements per dimension, and slicing extracts
sub-tensors:

In [ ]:
#  Tensor slicing — accessing sub-tensors
row_vector    = matrix[1]
column_vector = matrix[:, 1]
scalar        = matrix[0, 1]

print(f"`row_vector`:    {row_vector.numpy()}")
print(f"`column_vector`: {column_vector.numpy()}")
print(f"`scalar`:        {scalar.numpy()}")

#### What just happened — and what's missing

Tensors are TF's data container: fast, shape-safe, and GPU-ready. Every model input becomes
a tensor before processing.

**Missing piece**: We have the container, but we haven't built anything that *learns* yet.
For that we need operations TF can differentiate — next.

---

## Part 2 — Computations on Tensors

A convenient way to think about computations is in terms of **graphs**: tensors hold data,
and mathematical operations act on them in some order. TF eagerly executes operations (like
Python), while also recording them when inside a `tf.GradientTape` context.

Let's look at a simple example — adding two constants:

In [ ]:
#  Simple computation — node addition
a = tf.constant(15)
b = tf.constant(61)

c1 = tf.add(a, b)
c2 = a + b  # TF overrides "+" so it works on tensors
print(f"c1: {c1.numpy()}")
print(f"c2: {c2.numpy()}")

Now let's consider a slightly more complex example — a multi-step computation graph:

```
a, b → c = a+b, d = b-1 → e = c*d
```

Let's define this using TF operations:

In [ ]:
#  Multi-step computation graph: a,b → c,d → e
def func(a, b):
    c = tf.add(a, b)
    d = tf.subtract(b, tf.cast(1, b.dtype))
    e = tf.multiply(c, d)
    return e

Now call this function to execute the computation graph:

In [ ]:
#  Execute the computation graph
a = tf.constant(1.5)
b = tf.constant(2.5)
e_out = func(a, b)
print(f"e_out: {e_out.numpy()}")

#### What just happened — and what's missing

TF executed our computation and returned a tensor. But it did more than arithmetic — when
we later open a `tf.GradientTape` context and call `tape.gradient()`, TF will walk the
recorded operations in reverse to compute gradients for any `tf.Variable` used in the
computation.

**Missing piece**: We defined *what to compute*, but not *what to optimise*. For that we
need a trainable model with learnable weights — that's `tf.keras.layers.Layer`, next.

### Predict before you run

Before running the next cell, commit to an answer: using `w_size=0.15`, `w_age=-5.0`,
`bias=100.0`, what will the model predict for house [0] (1200 sq ft, 10 years, true price $250k)?

1. About **$180k** — the age penalty dominates.
2. About **$230k** — size and age partially cancel.
3. About **$310k** — size contribution dominates.

In [ ]:
#  Computation on our house dataset — price prediction by hand
size       = houses[0, 0]
age        = houses[0, 1]
true_price = prices[0]

w_size = tf.constant(0.15)
w_age  = tf.constant(-5.0)
bias   = tf.constant(100.0)

contrib_size = w_size * size
contrib_age  = w_age  * age
price_pred   = contrib_size + contrib_age + bias

print(f"House [0]: {size.numpy():.0f} sq ft, {age.numpy():.0f} years → true price ${true_price.numpy():.0f}k")
print(f"  w_size × size = {w_size.numpy():.2f} × {size.numpy():.0f} = ${contrib_size.numpy():.1f}k")
print(f"  w_age  × age  = {w_age.numpy():.2f} × {age.numpy():.0f} = ${contrib_age.numpy():.1f}k")
print(f"  + bias        = ${bias.numpy():.1f}k")
print(f"  → Predicted:  ${price_pred.numpy():.1f}k   (error: ${abs(price_pred.numpy() - true_price.numpy()):.1f}k)")
print("\nTF traced this computation. We'll use that trace for GradientTape next.")

### Exercise — manual weight tuning

**Predict**: Can you fit all 5 houses within ±$10k error just by adjusting numbers?

In [ ]:
# EXERCISE — manual weight tuning
# CHANGE these three weights to predict all 5 house prices within ±$10k error
w_size = tf.constant(0.15)   # ← try 0.10, 0.20, 0.25...
w_age  = tf.constant(-5.0)   # ← try -3.0, -8.0...
bias   = tf.constant(100.0)  # ← try 50.0, 150.0...

h_np, p_np = houses.numpy(), prices.numpy()
predictions = h_np[:, 0] * w_size.numpy() + h_np[:, 1] * w_age.numpy() + bias.numpy()
errors = np.abs(predictions - p_np)

print("Manual tuning results:")
print(f"  {'House':>6}  {'True':>8}  {'Predicted':>10}  {'Error':>8}")
for i, (p_true, p_pred, err) in enumerate(zip(p_np, predictions, errors)):
    check = "" if err < 10.0 else ""
    print(f"  [{i}]     ${p_true:6.1f}k   ${p_pred:6.1f}k      ${err:5.1f}k  {check}")
mean_error = errors.mean()
print(f"\nMean absolute error: ${mean_error:.1f}k")
if mean_error < 10.0:
    print("You nailed it by hand — but imagine 100 features, 10000 houses...")
else:
    print("→ Hand-tuning fails even on 5 houses. We need LEARNING: GradientTape + gradient descent.")

---

## Part 3 — Neural Networks in Keras

A perceptron does one thing: take a set of inputs, multiply each by a learned weight,
sum them up, then squash through a non-linearity: $y = \sigma(Wx + b)$.

Keras's [`tf.keras.layers.Layer`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Layer)
is the container that holds learnable weights (`tf.Variable`) and wires up the computation.
You subclass it, declare weights with `self.add_weight()`, and override `call()` with the
actual math. TF then traces every operation through those variables so `tape.gradient()`
can compute gradients for each one automatically.

**Key difference from PyTorch:** in `nn.Module` you declare weights as `nn.Parameter` and
define `forward()`. In Keras you use `add_weight()` and define `call()`.

#### **Predict first** — what does `tf.keras.layers.Layer` give us?

We're about to define a custom dense layer by subclassing `tf.keras.layers.Layer`. Predict
which is true:

1. We must manually call `tape.gradient()` on **each** weight separately inside the class.
2. Registering a tensor via `add_weight(..., trainable=True)` is sufficient — GradientTape
   tracks it automatically from that point forward.
3. We need a custom gradient-accumulation loop in `__init__` for weights to work.

In [ ]:
#  Hand-written dense layer (tf.keras.layers.Layer subclass)
class OurDenseLayer(tf.keras.layers.Layer):
    def __init__(self, num_inputs, num_outputs):
        super(OurDenseLayer, self).__init__()
        # add_weight registers a tf.Variable as a trainable parameter
        # — equivalent to nn.Parameter(torch.randn(...)) in PyTorch
        self.W = self.add_weight(
            shape=(num_inputs, num_outputs),
            initializer="random_normal",
            trainable=True,
            name="W",
        )
        self.bias = self.add_weight(
            shape=(num_outputs,),
            initializer="zeros",
            trainable=True,
            name="bias",
        )

    def call(self, x):
        z = tf.matmul(x, self.W) + self.bias
        y = tf.sigmoid(z)
        return y


print("OurDenseLayer defined — W and bias are add_weight() tensors, so GradientTape tracks them.")
print("No extra registration needed: add_weight(trainable=True) is sufficient.")

Now let's test the output of our layer:

In [ ]:
#  Test OurDenseLayer
num_inputs  = 2
num_outputs = 3
layer   = OurDenseLayer(num_inputs, num_outputs)
x_input = tf.constant([[1.0, 2.0]])
y       = layer(x_input)

print(f"input shape:  {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y.numpy()}")

#### What just happened — and what's missing

`OurDenseLayer` ran a single forward pass: random weights `W` multiplied the 2-feature
input, bias was added, sigmoid was applied, and a `(1, 3)` output tensor emerged.
Declaring weights with `add_weight(trainable=True)` is all that is required for
GradientTape to register them for gradient tracking — no extra bookkeeping.

What's missing: every new layer type needs its own subclass. Keras ships pre-built blocks
(`layers.Dense`, `layers.Activation`) that can be composed without writing a custom
`call()` — that is `tf.keras.Sequential`, shown next.

Conveniently, Keras has defined many pre-built layers including
[`layers.Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense) and
[`layers.Activation`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Activation).

We can use `tf.keras.Sequential` to stack layers like building blocks — the direct
equivalent of `nn.Sequential` in PyTorch:

In [ ]:
#  Same layer via tf.keras.Sequential
n_input_nodes  = 2
n_output_nodes = 3

# Note: layers.Dense does NOT need in_features — it infers from the first call
# PyTorch: nn.Linear(n_input_nodes, n_output_nodes) — in_features REQUIRED
model = tf.keras.Sequential([
    layers.Dense(n_output_nodes, activation="sigmoid"),
])
print(f"Sequential model defined — no input size needed (inferred on first call).")
print("Compare with PyTorch: nn.Sequential([nn.Linear(2, 3), nn.Sigmoid()]) — in_features required.")

In [ ]:
#  Test Sequential model
x_input      = tf.constant([[1.0, 2.0]])
model_output = model(x_input)  # first call builds the layer (infers input size)
print(f"input shape:   {x_input.shape}")
print(f"output shape:  {model_output.shape}")
print(f"output result: {model_output.numpy()}")

With Keras, we can also create flexible models by subclassing `tf.keras.layers.Layer`
or `tf.keras.Model`. This lets us define custom `call()` logic — equivalent to
PyTorch's `nn.Module` subclass with custom `forward()`:

In [ ]:
#  Custom call() via subclassing Layer
class LinearWithSigmoidActivation(tf.keras.layers.Layer):
    def __init__(self, num_outputs):
        super().__init__()
        self.linear     = layers.Dense(num_outputs)
        self.activation = layers.Activation("sigmoid")

    def call(self, inputs):
        linear_output = self.linear(inputs)
        output        = self.activation(linear_output)
        return output


print("LinearWithSigmoidActivation defined.")
print("Equivalent to tf.keras.Sequential([Dense, Activation]) but with explicit call().")
print("Subclassing lets you add branches, conditionals, or extra logic in call().")

In [ ]:
#  Test LinearWithSigmoidActivation
n_input_nodes  = 2
n_output_nodes = 3
model   = LinearWithSigmoidActivation(n_output_nodes)
x_input = tf.constant([[1.0, 2.0]])
y       = model(x_input)
print(f"input shape:  {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y.numpy()}")

### Three ways to define the same layer — which one, when?

| Approach | Pros | Cons |
| -------- | ---- | ---- |
| **`OurDenseLayer`** (manual `add_weight`) | Makes the "add_weight = register for gradient tracking" mechanism explicit | Most boilerplate; re-derive matmul + bias for every layer |
| **`tf.keras.Sequential`** | Fewest lines; layers stack top-to-bottom | Only works when data flows straight through in order — no branching |
| **Subclassing with `layers.Dense`** | Combines tested building blocks with full control over `call()` | More typing than Sequential for a simple stack |

`tf.keras.layers.Layer` affords a lot of flexibility. For example, we can use a boolean
argument in `call()` to specify different behaviors — equivalent to PyTorch's `isidentity`
pattern:

In [ ]:
#  Custom behavior: conditional identity pass
class LinearButSometimesIdentity(tf.keras.layers.Layer):
    def __init__(self, num_outputs):
        super().__init__()
        self.linear = layers.Dense(num_outputs)

    def call(self, inputs, isidentity=False):
        if isidentity:
            return inputs
        return self.linear(inputs)


print("LinearButSometimesIdentity defined.")
print("isidentity=True bypasses the Dense layer entirely — ordinary Python conditional.")

In [ ]:
#  Test LinearButSometimesIdentity
model   = LinearButSometimesIdentity(num_outputs=3)
x_input = tf.constant([[1.0, 2.0]])

out_with_linear   = model(x_input)
out_with_identity = model(x_input, isidentity=True)

print(f"input: {x_input.numpy()}")
print(f"network linear output:   {out_with_linear.numpy()}")
print(f"network identity output: {out_with_identity.numpy()}")

In [ ]:
#  HousePriceModel: same prediction, but with LEARNABLE weights
class HousePriceModel(tf.keras.Model):
    """Predicts house price from size and age using learnable tf.Variable weights."""

    def __init__(self):
        super().__init__()
        # tf.Variable is the Keras equivalent of nn.Parameter in PyTorch
        self.w_size = tf.Variable(tf.random.normal([1]) * 0.01, name="w_size")
        self.w_age  = tf.Variable(tf.random.normal([1]) * 0.01, name="w_age")
        self.bias   = tf.Variable(tf.random.normal([1]) * 0.01, name="bias")

    def call(self, size, age):
        return self.w_size * size + self.w_age * age + self.bias


tf.random.set_seed(42)
model = HousePriceModel()
print("Model parameters (random initialization):")
for var in model.trainable_variables:
    print(f"  {var.name}: {var.numpy()[0]:.4f}")

pred = model(houses[0, 0], houses[0, 1])
print(f"\nPrediction for house [0]: ${pred.numpy()[0]:.1f}k  (random weights → bad prediction)")
print("Next: we'll use GradientTape to LEARN better weights automatically.")

#### What just happened — and what's missing

`HousePriceModel` ran a single forward pass and produced a wrong prediction (random
weights). `tf.Variable` registers a tensor as mutable and trainable — GradientTape will
automatically track any computation that uses it.

**Missing piece**: The weights are randomly initialised — the model outputs nonsense.
We need a way to measure how wrong it is (**loss function**) and a way to systematically
improve the weights (**gradient descent**). That's `tf.GradientTape` — next.

---

## Part 4 — Automatic Differentiation

The core problem: we have a single loss number, and we need to know which direction to
nudge each weight to make it smaller. Computing those partial derivatives by hand — even
for our 3-weight house-price model — is tedious. For millions of weights it is impossible.

TensorFlow solves this with [`tf.GradientTape`](https://www.tensorflow.org/api_docs/python/tf/GradientTape):
any computation you perform inside a `with tf.GradientTape() as tape:` context is recorded.
When you call `tape.gradient(loss, variables)`, TF walks that record in reverse (applying
the chain rule layer by layer) and returns `∂loss/∂var` for each variable — in one pass.

**PyTorch equivalent:** `requires_grad=True` + `loss.backward()` + `var.grad`.
The mechanism is identical; only the API surface differs.

To see it at its clearest, let's verify it on $y = x^2$ — known analytic answer:

#### **Predict first** — what does `tape.gradient()` return?

We're about to compute the derivative of $y = x^2$ at $x = 3.0$ using GradientTape.
Predict the result:

1. `dy_dx = 9.0` — because $y = x^2 = 9$ at $x = 3$.
2. `dy_dx = 6.0` — because $\frac{dy}{dx} = 2x$, evaluated at $x = 3$.
3. `dy_dx = 3.0` — because $x = 3$.

In [ ]:
#  Gradient of y = x² at x = 3
# tf.Variable is watched automatically inside a GradientTape context
x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = x ** 2
dy_dx = tape.gradient(y, x)

print("dy_dx of y=x^2 at x=3.0 is:", dy_dx.numpy())
# dy/dx = 2x → at x=3: dy_dx = 6
assert float(dy_dx) == 6.0
print("  → tape.gradient() computed the analytic derivative automatically.")

The derivative alone does not train anything — you also need something to minimise.
In neural networks that is the **loss function**: a single number measuring how wrong
the current weights are.

To build the intuition without distractions, we will minimise $L = (x - x_f)^2$ — a
parabola with a known bottom at $x_f$. We deliberately will *not* solve it analytically;
instead, GradientTape computes $\partial L / \partial x$ at each iteration and gradient
descent steps toward the bottom. This is the **exact same loop** that trains every neural
network — just on one scalar instead of millions of weights.

### Key difference from PyTorch's gradient accumulation

In PyTorch, gradients **accumulate** in `.grad` tensors across calls to `.backward()` —
you must call `optimizer.zero_grad()` before each backward pass or the previous batch's
gradients pile on top.

**In TF/Keras, `tf.GradientTape` does not accumulate.** Each `with tf.GradientTape():`
block records a fresh computation from scratch, and `tape.gradient()` consumes and
discards the tape. There is no equivalent of `zero_grad()` — the problem doesn't exist.

### Predict before you run

Before running the next cell, commit to an answer: when minimising `L = (x - 4)²`
starting from a random `x`, where will gradient descent converge?

1. **x converges to 0** — gradient is always negative so x keeps decreasing.
2. **x converges to 4** — the global minimum, where the gradient is zero.
3. **x converges somewhere between start and 4** — never fully reaches the minimum.

In [ ]:
#  Gradient descent: minimize L = (x − x_f)²
tf.random.set_seed(42)
x_start = float(tf.random.normal([1]))
print(f"Initializing x={x_start:.4f}")

x   = tf.Variable(x_start)
x_f = 4
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-2)
history   = []

for i in range(500):
    with tf.GradientTape() as tape:
        loss = (x - x_f) ** 2
    grad = tape.gradient(loss, [x])
    optimizer.apply_gradients(zip(grad, [x]))
    history.append(float(x.numpy()))

plt.plot(history)
plt.plot([0, 500], [x_f, x_f])
plt.legend(("Predicted", "True"))
plt.xlabel("Iteration")
plt.ylabel("x value")
plt.title("Gradient descent converging to x_f = 4")
plt.show()
print(f"  → x converged to {history[-1]:.4f}; target was {x_f}.")
print("  → Same loop — GradientTape + optimizer.apply_gradients — trains every neural network.")

### Code Walkthrough: Gradient Descent on a Parabola

**`x = tf.Variable(x_start)` — mutable tensor, watched automatically**
`tf.Variable` is mutable and watched by `GradientTape` automatically. No `requires_grad=True`
flag needed. PyTorch: `x = torch.tensor([x], requires_grad=True)` re-wrapped each step.

**`with tf.GradientTape() as tape:` — open the gradient recorder**
Every operation inside this context on watched variables is recorded. The tape is
**consumed** on `tape.gradient()` — no `zero_grad()` needed on the next iteration;
a fresh `with` block starts a fresh recording.

**`tape.gradient(loss, [x])` — compute ∂loss/∂x automatically**
For `loss = (x - x_f)²`, the analytic gradient is `2(x - x_f)`. TF computes this via
the recorded computation. The result is a list of gradients, one per variable.

**`optimizer.apply_gradients(zip(grad, [x]))` — apply the update**
The SGD update: $x \leftarrow x - \alpha \cdot \nabla_x L$. After 500 steps, `x` converges
to `x_f = 4`.

In [ ]:
#  Visualizing what tape.gradient() actually does
tf.random.set_seed(42)
model_viz = HousePriceModel()
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-5)

with tf.GradientTape() as tape:
    preds = model_viz(houses[:, 0], houses[:, 1])
    loss  = tf.reduce_mean((preds - prices) ** 2)

grads = tape.gradient(loss, model_viz.trainable_variables)
print("After tape.gradient(), gradients computed:")
for var, grad in zip(model_viz.trainable_variables, grads):
    print(f"  ∂loss/∂{var.name}: {float(grad.numpy()[0]):+.4f}")

print(f"\nLoss: {loss.numpy():.2f}")
print("→ tape.gradient() computed all three gradients automatically.")

print(f"\nBefore update: w_size = {float(model_viz.w_size.numpy()[0]):.4f}")
optimizer.apply_gradients(zip(grads, model_viz.trainable_variables))
print(f"After step:    w_size = {float(model_viz.w_size.numpy()[0]):.4f}  (moved toward lower loss)")

### Training Loop to Convergence

One `optimizer.apply_gradients()` nudges every weight a tiny bit — barely visible in a
single printout. Real training repeats **GradientTape → loss → gradient → step** hundreds
of times. Let's run `HousePriceModel` to convergence and watch the loss curve.

In [ ]:
#  Full training loop: iterate to convergence
tf.random.set_seed(42)
model_trained = HousePriceModel()
optimizer     = tf.keras.optimizers.SGD(learning_rate=1e-5)

loss_history = []
for epoch in range(500):
    with tf.GradientTape() as tape:
        preds = model_trained(houses[:, 0], houses[:, 1])
        loss  = tf.reduce_mean((preds - prices) ** 2)
    grads = tape.gradient(loss, model_trained.trainable_variables)
    optimizer.apply_gradients(zip(grads, model_trained.trainable_variables))
    loss_history.append(float(loss.numpy()))
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  loss = {float(loss.numpy()):8.2f}")
print(f"epoch {len(loss_history)-1:3d}  loss = {loss_history[-1]:8.2f}  (final)")

plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss ($k²)")
plt.title("HousePriceModel training loop — loss vs. epoch")
plt.show()

print("\nFinal predictions vs. true prices:")
final_preds = model_trained(houses[:, 0], houses[:, 1])
for i, (p_true, p_pred) in enumerate(zip(prices.numpy(), final_preds.numpy())):
    print(f"  house [{i}]  true=${p_true:.0f}k  predicted=${p_pred[0]:.0f}k")

print(f"\n→ {len(loss_history)} iterations of GradientTape → loss → apply_gradients took the loss from "
      f"{loss_history[0]:.1f} to {loss_history[-1]:.1f}.")

#### What just happened — from toy to production

We trained a 3-parameter model with `tf.GradientTape` for 500 full iterations and watched
the loss curve fall toward zero. The same mechanism trains:

- GPT-2: 1.5 billion parameters
- Stable Diffusion: 860 million parameters

**The only difference is scale.** The GradientTape, the gradient computation, the optimizer
step — all identical.

---

## Part 5 — From Toy to Production: Same Machinery, Bigger Numbers

Our house-price model has **3 learnable parameters**. A production deep-learning model
has millions — but the *mechanism* is identical: `tf.Variable`, `call()`, `GradientTape`,
`apply_gradients`.

| Model | Parameters | Same GradientTape? | Same tf.Variable? |
| ----- | ---------- | ------------------ | ----------------- |
| Our toy (house prices) | 3 | ✓ | ✓ |
| ResNet50 (image classification) | 25.6 million | ✓ | ✓ |
| GPT-2 (language model) | 1.5 billion | ✓ | ✓ |
| Stable Diffusion | 860 million | ✓ | ✓ |

**The only difference is scale.**

### Predict before you run

Our house-price model has 3 learnable parameters. Roughly how many does ResNet50 have?

1. About **25 thousand** — still small, just deeper.
2. About **25 million** — three orders of magnitude more than our toy.
3. About **25 billion** — in the same range as large language models.

In [ ]:
#  Toy-to-real: same tf.keras.Model, different scale
try:
    real_model = tf.keras.applications.ResNet50(
        weights=None, input_shape=(224, 224, 3), classes=1000
    )
    total     = real_model.count_params()
    trainable = sum(tf.size(v).numpy() for v in real_model.trainable_variables)
    print(f"ResNet50 architecture:")
    print(f"  Total parameters:     {total:,}")
    print(f"  Trainable parameters: {trainable:,}")
    print(f"  Layers: {len(real_model.layers)}")
    print("\nEvery one of those 25.6M parameters is a tf.Variable, just like our w_size.")
    print("Every training step uses GradientTape. The machinery you learned on 3 weights")
    print("scales to millions — no new concepts needed.")
except Exception as e:
    print(f"[{e}]")
    print("The point stands: ResNet50 uses the same tf.Variable / GradientTape machinery")
    print("you just learned on 5 houses and 3 weights.")

---

## Part 6 — Sequence Modeling with SimpleRNN and LSTM

Parts 1–5 covered the complete Keras mental model: tensors, computation, `Layer`/`Model`,
`GradientTape`, and gradient descent. Every architecture so far accepted **fixed-size inputs**.

When inputs are **variable-length sequences** — sentences, melodies, time series — we need
a different structure: one that carries **state** across time steps.

### Why sequences need recurrent networks

A dense layer treats each input independently — there is no concept of "what came before".
A recurrent layer maintains a **hidden state** $h_t$ that is updated at each time step:

$$h_t = f(W_{ih} \cdot x_t + W_{hh} \cdot h_{t-1} + b)$$

The hidden state acts as a compressed memory of the sequence so far. After processing all
$T$ steps, $h_T$ summarises the entire sequence.

### SimpleRNN vs LSTM

| | SimpleRNN | LSTM |
| -- | --------- | ---- |
| State | Single hidden state $h_t$ | Cell state $c_t$ + hidden state $h_t$ |
| Gates | None | Input, forget, output gates |
| Long-range memory | Poor (vanishing gradients) | Good (gates control memory retention) |
| Use when | Toy problems, short sequences | Real tasks, longer sequences |

Both use the same Keras API; just replace `layers.SimpleRNN` with `layers.LSTM`.

In [ ]:
#  Generate sine wave sequence data
np.random.seed(42)

t      = np.linspace(0, 8 * np.pi, 2000)
signal = np.sin(t).astype(np.float32)

# Build (input sequence, next value) pairs
SEQ_LEN = 50
X = np.array([signal[i : i + SEQ_LEN] for i in range(len(signal) - SEQ_LEN)])
y = signal[SEQ_LEN:]

# Reshape for RNN: (samples, timesteps, features)
X = X.reshape(X.shape[0], SEQ_LEN, 1)
y = y.reshape(-1, 1)

# Train / test split
split   = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"X shape: {X.shape}  → (samples, timesteps, features)")
print(f"y shape: {y.shape}  → (samples, 1) — one next-value per sequence")
print(f"Train: {X_train.shape[0]} sequences | Test: {X_test.shape[0]} sequences")

plt.plot(signal[:200], label="sine wave")
plt.title("First 200 samples of the sine wave signal")
plt.xlabel("Time step")
plt.ylabel("Amplitude")
plt.legend()
plt.show()

### Define a SimpleRNN model

`layers.SimpleRNN(units)` processes the input sequence one step at a time, maintaining
a hidden state of size `units`. With `return_sequences=False` (default), it returns only
the final hidden state — a fixed-size vector summarising the whole sequence.

In [ ]:
#  Build and compile a SimpleRNN model
tf.random.set_seed(42)

rnn_model = tf.keras.Sequential(
    [
        keras.Input(shape=(SEQ_LEN, 1)),
        layers.SimpleRNN(32, return_sequences=False),
        layers.Dense(1),
    ]
)
rnn_model.compile(optimizer="adam", loss="mse")
rnn_model.summary()
print("\nSimpleRNN: processes SEQ_LEN=50 steps, emits one 32-d hidden state, Dense maps to 1.")

In [ ]:
#  Train the SimpleRNN
rnn_history = rnn_model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1,
)

In [ ]:
#  Plot SimpleRNN loss curves
plt.plot(rnn_history.history["loss"],     label="SimpleRNN train")
plt.plot(rnn_history.history["val_loss"], label="SimpleRNN val")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("SimpleRNN Training Loss — Sine Wave Prediction")
plt.legend()
plt.show()

### Define an LSTM model

`layers.LSTM(units)` adds gating mechanisms: an **input gate** controls what new
information to store, a **forget gate** decides what to erase, and an **output gate**
determines the hidden state. Together they allow the model to retain information across
much longer sequences than a SimpleRNN.

The API is identical to `SimpleRNN` — just swap the layer name.

In [ ]:
#  Build and compile an LSTM model
tf.random.set_seed(42)

lstm_model = tf.keras.Sequential(
    [
        keras.Input(shape=(SEQ_LEN, 1)),
        layers.LSTM(64, return_sequences=False),
        layers.Dense(1),
    ]
)
lstm_model.compile(optimizer="adam", loss="mse")
lstm_model.summary()
print("\nLSTM: same API as SimpleRNN — just replace the layer. 4× more parameters due to gates.")

In [ ]:
#  Train the LSTM
lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1,
)

In [ ]:
#  Compare loss curves: SimpleRNN vs LSTM
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rnn_history.history["loss"],      label="train")
axes[0].plot(rnn_history.history["val_loss"],  label="val")
axes[0].set_title("SimpleRNN")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()

axes[1].plot(lstm_history.history["loss"],     label="train")
axes[1].plot(lstm_history.history["val_loss"], label="val")
axes[1].set_title("LSTM")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE")
axes[1].legend()

plt.suptitle("SimpleRNN vs LSTM: training loss on sine wave prediction", fontweight="bold")
plt.tight_layout()
plt.show()

rnn_final  = rnn_history.history["val_loss"][-1]
lstm_final = lstm_history.history["val_loss"][-1]
print(f"Final val MSE — SimpleRNN: {rnn_final:.5f} | LSTM: {lstm_final:.5f}")

In [ ]:
#  Prediction visualization: LSTM on test set
lstm_preds = lstm_model.predict(X_test, verbose=0)

n_plot = 200
plt.figure(figsize=(12, 4))
plt.plot(y_test[:n_plot],          label="True",      alpha=0.8)
plt.plot(lstm_preds[:n_plot],      label="LSTM pred", alpha=0.8, linestyle="--")
plt.xlabel("Test sequence index")
plt.ylabel("Amplitude")
plt.title(f"LSTM predictions on sine wave (first {n_plot} test steps)")
plt.legend()
plt.show()
print("The LSTM learns to predict the next sine wave value from a window of 50 past values.")

#### What just happened

We built and trained two sequence models:

1. **SimpleRNN** — a basic recurrent layer with a single hidden state. Fine for short sequences
   but suffers from vanishing gradients on longer ones.
2. **LSTM** — adds input, forget, and output gates that control information flow through the
   cell state. Better at retaining long-range context.

Both used the same `model.compile()` + `model.fit()` workflow from the primer. Under the
hood, `model.fit()` is running the same `tf.GradientTape` → `tape.gradient()` →
`apply_gradients()` loop you built manually in Part 4 — just abstracted away.

The transition from `model.fit()` (high-level) to `tf.GradientTape` (low-level) gives you
full control over:
- What loss you compute (custom losses, multi-task objectives)
- How gradients are modified before applying (gradient clipping)
- How you process sequences (teacher forcing, scheduled sampling)

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated

**Tensors:**
- Creation — `tf.constant`, `tf.Variable`, `tf.zeros`, `tf.random.normal`
- Properties — `.shape`, `.ndim`, `.numpy()` for converting to NumPy
- Indexing/slicing — row, column, scalar extraction
- NumPy ↔ TF interop — `tf.constant(np_array)`, `.numpy()`
- Shape-mismatch safety — exception raised at the point of the mistake
- GPU availability — `tf.config.list_physical_devices('GPU')`

**Computation & Autograd:**
- TF ops — `tf.add`, `tf.subtract`, `tf.multiply`, `tf.matmul`
- `tf.GradientTape` — recording, `tape.gradient()`, consuming the tape
- Manual gradient descent on $L = (x - x_f)^2$, iterated to convergence
- `tf.keras.optimizers.SGD` / `Adam` — `apply_gradients(zip(grads, vars))`

**Model Building:**
- `add_weight()` + hand-written `call()` (`OurDenseLayer`)
- `tf.keras.Sequential` with `layers.Dense`
- Subclassing `tf.keras.layers.Layer` with custom logic
- `tf.keras.Model` with `tf.Variable` weights (`HousePriceModel`)

**Training Loop:**
- `HousePriceModel` trained to convergence over 500 epochs, loss curve plotted

**Sequence Modeling:**
- `layers.SimpleRNN` — basic recurrent unit, trained on sine wave prediction
- `layers.LSTM` — gated recurrent unit, compared against SimpleRNN
- Side-by-side loss curve comparison and prediction visualization

### Tier 2 — Explained but Not Fully Implemented
- GPU tensor routing — TF uses GPU automatically; not demonstrated because no GPU in this env
- Non-convex loss surfaces / local minima — explained in prose, parabola is convex special case

### Tier 3 — Named but Out of Scope
- Tensor manipulation — `tf.reshape`, `tf.transpose`, `tf.squeeze`, broadcasting rules
- Advanced autograd — `tf.custom_gradient`, higher-order gradients, gradient clipping
- Mini-batch training — `tf.data.Dataset`, shuffling, train/val splits, regularization
- BPTT depth analysis — vanishing/exploding gradients in deep unrolled RNNs
- Attention — transformer self-attention, positional encoding (Lab 2)
- Model checkpointing — `tf.train.Checkpoint`, `ModelCheckpoint` callback
- Mixed precision, `@tf.function`, deployment/export

---

## Summary — What You Built

| Step | Concept | Key Idea |
| ---- | ------- | -------- |
| 1 | Tensors as Data Containers | Scalars → vectors → matrices → batches; faster + safer than lists |
| 2 | Operations on Tensors | TF builds computation graphs automatically |
| 3 | The Manual Prediction Problem | Hand-tuning 3 weights fails; 100 features is impossible |
| 4 | Neural Networks in Keras | `tf.keras.layers.Layer` wraps learnable `tf.Variable` weights |
| 5 | Automatic Differentiation | `tape.gradient()` computes ∂loss/∂every_weight in one call |
| 6 | Gradient Descent in Action | Iterative `apply_gradients` drives loss toward zero |
| 7 | From Toy to Production | Same GradientTape scales from 3 params to 25M (ResNet50) |
| 8 | Sequence Modeling | `layers.SimpleRNN` and `layers.LSTM` extend the pattern to ordered sequences |

### Key Insights to Keep

- **Tensors are purpose-built**: GPU-ready, shape-safe, faster than lists for matrix ops.
- **`tf.keras.layers.Layer` = learnable weights + forward pass**: `add_weight()` registers
  variables for automatic gradient tracking — no extra bookkeeping.
- **GradientTape is automatic calculus**: `tape.gradient()` computes every ∂loss/∂var in
  one call, no manual derivatives needed.
- **No `zero_grad()` needed**: GradientTape records a fresh computation on every `with`
  block — no gradient accumulation problem unlike PyTorch.
- **SimpleRNN → LSTM is a one-word swap**: the API is identical; the internal mechanism
  gains gating for better long-range memory.
- **`model.fit()` wraps the GradientTape loop**: every epoch of `model.fit()` runs
  `tape.gradient()` + `apply_gradients()` internally for each batch.

---

## When to Use What — Keras Patterns from This Notebook

| Situation | Pattern to reach for | Why |
| --------- | -------------------- | --- |
| Building a custom layer with learnable weights | `tf.keras.layers.Layer` subclass with `add_weight()` | Registers weights for gradient tracking automatically |
| Simple stack of standard layers | `tf.keras.Sequential` | Less boilerplate; no custom `call()` needed |
| Layer with conditional logic or skip connections | `tf.keras.Model` subclass with explicit `call()` | `Sequential` can't branch |
| Full manual control over the training step | `tf.GradientTape` custom loop | When `model.fit()` doesn't expose what you need |
| Sequence prediction from ordered inputs | `layers.SimpleRNN` or `layers.LSTM` | Carries hidden state across time steps |
| Long sequences, long-range memory | `layers.LSTM` | Gating prevents vanishing gradients |
| Monitoring convergence during manual training | Append `loss.numpy()` to a list each epoch; plot after | Equivalent to `history` from `model.fit()` |
| Running the same code on CPU and GPU | Nothing to change — TF routes automatically | Unlike PyTorch which requires explicit `.to(device)` |

## Sequence Tensor Contract

This notebook uses tabular tensors: `(batch, features)` for the house-price model,
`(batch, timesteps, features)` for the RNN/LSTM models.

Keep this contract in view:
- `(batch, timesteps, features)` — a batch of sequences, each of length `timesteps`
  with `features` values per step
- A model with `return_sequences=False` outputs `(batch, units)` — one summary per sequence
- A model with `return_sequences=True` outputs `(batch, timesteps, units)` — one output per step
- For variable-length sequences, padding and masking identify which steps are real

## Next Module: Deeper Sequence Models and Transformers

This is a TensorFlow/Keras fundamentals notebook. It builds `layers.SimpleRNN` and
`layers.LSTM` from first principles but does not cover:

- Stacked (multi-layer) RNNs with `return_sequences=True` feeding into the next layer
- Bidirectional RNNs with `layers.Bidirectional`
- Encoder–decoder architectures for sequence-to-sequence tasks
- Attention mechanisms and transformer self-attention

Next, use the sequence tensor contract `(batch, timesteps, features)` to build deeper
recurrent models, then compare their sequential memory path with transformer self-attention,
where each token can weigh all other positions directly. The same `tf.keras.Model`,
`GradientTape`, and optimizer pattern from this notebook remains in place — the model's
data flow is what changes.